# Minimum Spanning Tree Algorithms

This notebook is designed for interactive use in a bachelor-level algorithms class.

It assumes basic familiarity with graphs, trees, and connected components. It does **not** assume prior knowledge of shortest path algorithms.

## Learning goals
- Define spanning trees and minimum spanning trees (MSTs).
- Explain why greedy choices can work for MST.
- Trace Kruskal's and Prim's algorithms on the same graph.
- See why MSTs are sometimes unique and sometimes not.
- Contrast MST with a shortest-path tree without assuming any shortest-path algorithm.

## Colab note
The setup cell below installs `networkx` and `matplotlib` only if they are missing, so the notebook should run on Google Colab without any repo-specific files.


In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

for package in ("networkx", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import heapq
import matplotlib.pyplot as plt
import networkx as nx

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.facecolor"] = "#fafafa"
plt.rcParams["font.size"] = 11

print(f"networkx {nx.__version__}")


In [ ]:
def edge_key(u: str, v: str) -> tuple[str, str]:
    return tuple(sorted((u, v)))


def canonicalize_edges(edges: list[tuple[str, str, int]]) -> list[tuple[str, str, int]]:
    return sorted(
        [(edge_key(u, v)[0], edge_key(u, v)[1], w) for u, v, w in edges],
        key=lambda item: (item[2], item[0], item[1]),
    )


def vertices_from_edges(edges: list[tuple[str, str, int]]) -> list[str]:
    return sorted({u for u, _, _ in edges} | {v for _, v, _ in edges})


def total_weight(edges: list[tuple[str, str, int]]) -> int:
    return sum(w for _, _, w in edges)


def build_graph(edges: list[tuple[str, str, int]]) -> nx.Graph:
    graph = nx.Graph()
    for u, v, w in edges:
        graph.add_edge(u, v, weight=w)
    return graph


def adjacency_from_edges(edges: list[tuple[str, str, int]]) -> dict[str, list[tuple[int, str, str]]]:
    adjacency = {vertex: [] for vertex in vertices_from_edges(edges)}
    for u, v, w in edges:
        adjacency[u].append((w, u, v))
        adjacency[v].append((w, v, u))
    return adjacency


def route_lengths_in_tree(tree_edges: list[tuple[str, str, int]], start: str) -> dict[str, int]:
    tree = build_graph(tree_edges)
    return dict(nx.shortest_path_length(tree, source=start, weight="weight"))


def draw_weighted_graph(
    edges: list[tuple[str, str, int]],
    pos: dict[str, tuple[float, float]] | None = None,
    tree_edges: list[tuple[str, str, int]] | None = None,
    title: str = "",
    ax=None,
):
    graph = build_graph(edges)
    if ax is None:
        _, ax = plt.subplots()
    if pos is None:
        pos = nx.spring_layout(graph, seed=7)

    highlighted = {edge_key(u, v) for u, v, _ in (tree_edges or [])}
    edge_colors = []
    widths = []
    for u, v in graph.edges():
        if edge_key(u, v) in highlighted:
            edge_colors.append("#d1495b")
            widths.append(3.2)
        else:
            edge_colors.append("#9aa0a6")
            widths.append(1.8)

    nx.draw_networkx_nodes(
        graph,
        pos,
        node_color="#f4e285",
        edgecolors="#444444",
        node_size=1200,
        ax=ax,
    )
    nx.draw_networkx_labels(graph, pos, font_weight="bold", ax=ax)
    nx.draw_networkx_edges(graph, pos, edge_color=edge_colors, width=widths, ax=ax)
    nx.draw_networkx_edge_labels(
        graph,
        pos,
        edge_labels={(u, v): data["weight"] for u, v, data in graph.edges(data=True)},
        font_color="#333333",
        ax=ax,
    )
    if title:
        ax.set_title(title)
    ax.axis("off")
    return ax


def print_tree(label: str, tree_edges: list[tuple[str, str, int]]) -> None:
    print(f"{label}: {canonicalize_edges(tree_edges)}")
    print(f"total weight = {total_weight(tree_edges)}")


## Problem Statement

A **spanning tree** of a connected, undirected graph is a subgraph that:
- uses every vertex,
- is connected,
- has no cycles.

A **minimum spanning tree** is a spanning tree whose total edge weight is as small as possible.

We will focus on this question:

> How can we connect every vertex as cheaply as possible without creating cycles?


In [ ]:
sample_edges = [
    ("A", "B", 4),
    ("A", "C", 2),
    ("B", "C", 1),
    ("B", "D", 6),
    ("C", "D", 8),
    ("C", "E", 10),
    ("D", "E", 3),
    ("D", "F", 7),
    ("E", "F", 5),
    ("C", "F", 9),
]

sample_pos = {
    "A": (0.0, 1.0),
    "B": (1.3, 2.2),
    "C": (1.5, 0.2),
    "D": (3.5, 2.0),
    "E": (4.6, 0.3),
    "F": (5.8, 1.5),
}

draw_weighted_graph(sample_edges, pos=sample_pos, title="Weighted graph used throughout the notebook")
plt.show()


## Warm-up: What If All Edge Weights Are Equal?

Every spanning tree on `n` vertices has exactly `n - 1` edges.

So if every edge has the same weight `c`, then **every spanning tree has total weight `c(n - 1)`**. In that special case, any spanning tree is automatically minimum.

Instructor prompt:
Ask students why a cycle is never needed in a spanning tree.


In [ ]:
equal_weight_edges = [
    ("A", "B", 1),
    ("B", "C", 1),
    ("C", "D", 1),
    ("D", "A", 1),
    ("A", "C", 1),
]

equal_weight_pos = {
    "A": (0.0, 1.0),
    "B": (1.0, 2.0),
    "C": (2.0, 1.0),
    "D": (1.0, 0.0),
}

tree_a = [("A", "B", 1), ("B", "C", 1), ("C", "D", 1)]
tree_b = [("A", "D", 1), ("D", "C", 1), ("A", "B", 1)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
draw_weighted_graph(equal_weight_edges, pos=equal_weight_pos, title="All edges have weight 1", ax=axes[0])
draw_weighted_graph(equal_weight_edges, pos=equal_weight_pos, tree_edges=tree_a, title=f"Spanning tree A (weight {total_weight(tree_a)})", ax=axes[1])
draw_weighted_graph(equal_weight_edges, pos=equal_weight_pos, tree_edges=tree_b, title=f"Spanning tree B (weight {total_weight(tree_b)})", ax=axes[2])
plt.show()


## Greedy Ideas for MST

Two useful ideas drive most MST algorithms:

1. **Cut idea**: if we split the vertices into two groups, the cheapest edge crossing that split is safe to take.
2. **Cycle idea**: if an edge is the heaviest edge on a cycle, we do not need it in an MST.

Kruskal's algorithm mostly feels like "avoid bad cycle edges".
Prim's algorithm mostly feels like "grow the tree through the cheapest frontier edge".

We will use these ideas informally in class. Full proofs can be added later if needed.


## Kruskal's Algorithm

Kruskal sorts the edges from cheapest to most expensive, then scans them in that order.

For each edge:
- take it if it connects two different connected components,
- skip it if it would create a cycle.

To test connected components efficiently, we use a **disjoint-set union** (DSU), also called **union-find**.

Instructor prompt:
Before running the next cell, ask students which edge Kruskal must take first on the sample graph.


In [ ]:
class DisjointSet:
    def __init__(self, items: list[str]):
        self.parent = {item: item for item in items}
        self.rank = {item: 0 for item in items}

    def find(self, item: str) -> str:
        if self.parent[item] != item:
            self.parent[item] = self.find(self.parent[item])
        return self.parent[item]

    def union(self, left: str, right: str) -> bool:
        left_root = self.find(left)
        right_root = self.find(right)
        if left_root == right_root:
            return False

        if self.rank[left_root] < self.rank[right_root]:
            left_root, right_root = right_root, left_root

        self.parent[right_root] = left_root
        if self.rank[left_root] == self.rank[right_root]:
            self.rank[left_root] += 1
        return True


def kruskal_mst(edges: list[tuple[str, str, int]], debug: bool = False) -> list[tuple[str, str, int]]:
    vertices = vertices_from_edges(edges)
    dsu = DisjointSet(vertices)
    mst = []

    sorted_edges = sorted(edges, key=lambda edge: (edge[2], edge_key(edge[0], edge[1])))

    for u, v, w in sorted_edges:
        if dsu.union(u, v):
            mst.append((u, v, w))
            if debug:
                print(f"take {u}-{v} (weight {w})")
        elif debug:
            print(f"skip {u}-{v} (weight {w}) because it would create a cycle")

        if len(mst) == len(vertices) - 1:
            break

    if len(mst) != len(vertices) - 1:
        raise ValueError("Graph must be connected to have a spanning tree")

    return mst


In [ ]:
kruskal_tree = kruskal_mst(sample_edges, debug=True)
print()
print_tree("Kruskal MST", kruskal_tree)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
draw_weighted_graph(sample_edges, pos=sample_pos, title="Original graph", ax=axes[0])
draw_weighted_graph(sample_edges, pos=sample_pos, tree_edges=kruskal_tree, title="Edges chosen by Kruskal", ax=axes[1])
plt.show()


### Why DSU matters

Kruskal spends most of its time in two places:
- sorting the edges,
- checking whether an edge joins two different components.

With sorting plus union-find, the running time is:

- `O(E log E)`
- often written as `O(E log V)` because `E <= V^2`

Kruskal is often a good fit for sparse graphs because it works directly with the edge list.


## Uniqueness and Ties

If all edge weights are distinct, the MST is unique.

If some edge weights tie, then there can be several different MSTs with the same total weight. The answer is still correct, but it may not be the only correct answer.


In [ ]:
tie_edges = [
    ("A", "B", 1),
    ("B", "C", 1),
    ("C", "D", 1),
    ("D", "A", 1),
    ("A", "C", 2),
    ("B", "D", 2),
]

tie_pos = {
    "A": (0.0, 1.0),
    "B": (1.0, 2.0),
    "C": (2.0, 1.0),
    "D": (1.0, 0.0),
}

tie_tree_1 = [("A", "B", 1), ("B", "C", 1), ("C", "D", 1)]
tie_tree_2 = [("A", "D", 1), ("D", "C", 1), ("A", "B", 1)]

print_tree("MST candidate 1", tie_tree_1)
print_tree("MST candidate 2", tie_tree_2)
print()
print_tree("Deterministic Kruskal output", kruskal_mst(tie_edges))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
draw_weighted_graph(tie_edges, pos=tie_pos, title="Graph with ties", ax=axes[0])
draw_weighted_graph(tie_edges, pos=tie_pos, tree_edges=tie_tree_1, title="One valid MST", ax=axes[1])
draw_weighted_graph(tie_edges, pos=tie_pos, tree_edges=tie_tree_2, title="Another valid MST", ax=axes[2])
plt.show()


## Prim's Algorithm

Prim grows a single tree instead of building a forest.

Basic idea:
1. Start from one vertex.
2. Look at all edges leaving the current tree.
3. Take the cheapest one that reaches a new vertex.
4. Repeat until every vertex is included.

Prim is **not** a shortest-path algorithm. It does not try to minimize the route from the start to every other vertex. It tries to minimize the **total weight of the final tree**.

Instructor prompt:
Ask students which edges are currently on the frontier after starting from `A`.


In [ ]:
def prim_mst(
    edges: list[tuple[str, str, int]],
    start: str | None = None,
    debug: bool = False,
) -> list[tuple[str, str, int]]:
    vertices = vertices_from_edges(edges)
    if not vertices:
        return []

    adjacency = adjacency_from_edges(edges)
    if start is None:
        start = vertices[0]
    if start not in adjacency:
        raise ValueError(f"Unknown start vertex: {start}")

    visited = {start}
    heap = []
    mst = []

    for w, u, v in adjacency[start]:
        heapq.heappush(heap, (w, edge_key(u, v), u, v))

    if debug:
        print(f"start at {start}")

    while heap and len(visited) < len(vertices):
        w, _, u, v = heapq.heappop(heap)
        if v in visited:
            continue

        visited.add(v)
        mst.append((u, v, w))
        if debug:
            print(f"take {u}-{v} (weight {w})")

        for next_w, _, next_v in adjacency[v]:
            if next_v not in visited:
                heapq.heappush(heap, (next_w, edge_key(v, next_v), v, next_v))

    if len(mst) != len(vertices) - 1:
        raise ValueError("Graph must be connected to have a spanning tree")

    return mst


In [ ]:
prim_tree = prim_mst(sample_edges, start="A", debug=True)
print()
print_tree("Prim MST from A", prim_tree)
print()
print(f"Kruskal and Prim have the same total weight? {total_weight(kruskal_tree) == total_weight(prim_tree)}")


In [ ]:
prim_from_a = prim_mst(tie_edges, start="A")
prim_from_c = prim_mst(tie_edges, start="C")

print_tree("Prim on tie graph from A", prim_from_a)
print_tree("Prim on tie graph from C", prim_from_c)
print()
print("Same edge set?", canonicalize_edges(prim_from_a) == canonicalize_edges(prim_from_c))
print("On this graph, our deterministic tie-breaking makes both starts choose the same MST.")
print("The previous cell already showed that the graph still has multiple valid MSTs.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
draw_weighted_graph(tie_edges, pos=tie_pos, tree_edges=prim_from_a, title="Prim from A", ax=axes[0])
draw_weighted_graph(tie_edges, pos=tie_pos, tree_edges=prim_from_c, title="Prim from C", ax=axes[1])
plt.show()


### Kruskal vs Prim

Both algorithms are greedy and both always produce an MST on a connected, undirected weighted graph.

Typical intuition:
- **Kruskal** grows a forest and keeps merging components.
- **Prim** grows one tree outward from a starting vertex.

Typical implementation costs:
- **Kruskal + DSU**: `O(E log E)`
- **Prim + heap + adjacency lists**: `O(E log V)`

In practice, the better choice often depends on how the graph is stored and whether it is sparse or dense.


## Optional Enrichment: Boruvka's Algorithm

Boruvka's algorithm is historically important and still appears inside fast MST methods.

It starts with every vertex as its own component. In each round, every component chooses its cheapest outgoing edge, and then all those chosen edges are added at once.

This section is optional for a first pass through the topic. Kruskal and Prim are the main lecture algorithms.


In [ ]:
def boruvka_mst(edges: list[tuple[str, str, int]], debug: bool = False) -> list[tuple[str, str, int]]:
    vertices = vertices_from_edges(edges)
    dsu = DisjointSet(vertices)
    mst = []
    component_count = len(vertices)
    sorted_edges = sorted(edges, key=lambda edge: (edge[2], edge_key(edge[0], edge[1])))

    while component_count > 1:
        cheapest: dict[str, tuple[str, str, int]] = {}

        for u, v, w in sorted_edges:
            root_u = dsu.find(u)
            root_v = dsu.find(v)
            if root_u == root_v:
                continue

            candidate = (u, v, w)
            if root_u not in cheapest or (w, edge_key(u, v)) < (cheapest[root_u][2], edge_key(cheapest[root_u][0], cheapest[root_u][1])):
                cheapest[root_u] = candidate
            if root_v not in cheapest or (w, edge_key(u, v)) < (cheapest[root_v][2], edge_key(cheapest[root_v][0], cheapest[root_v][1])):
                cheapest[root_v] = candidate

        if not cheapest:
            raise ValueError("Graph must be connected to have a spanning tree")

        for u, v, w in cheapest.values():
            if dsu.union(u, v):
                mst.append((u, v, w))
                component_count -= 1
                if debug:
                    print(f"merge with {u}-{v} (weight {w})")

    return mst


In [ ]:
boruvka_tree = boruvka_mst(sample_edges, debug=True)
print()
print_tree("Boruvka MST", boruvka_tree)
print()
print(f"Kruskal total weight: {total_weight(kruskal_tree)}")
print(f"Prim total weight:    {total_weight(prim_tree)}")
print(f"Boruvka total weight: {total_weight(boruvka_tree)}")


## MST Is Not the Same as a Shortest-Path Tree

Even if students have not yet seen shortest-path algorithms, the distinction is worth stating now.

- An **MST** minimizes the **sum of all edge weights in the tree**.
- A **shortest-path tree from source `s`** minimizes the route from `s` to each vertex individually.

Those are different optimization goals, so the best trees can be different.

In the next cell we compare two finished trees. We are **not** running a shortest-path algorithm here; we are just comparing the goals.


In [ ]:
contrast_edges = [
    ("A", "B", 2),
    ("A", "C", 2),
    ("A", "D", 2),
    ("B", "C", 1),
    ("C", "D", 1),
]

contrast_pos = {
    "A": (0.0, 1.0),
    "B": (1.4, 2.0),
    "C": (2.6, 1.0),
    "D": (3.8, 0.0),
}

mst_example = [("A", "B", 2), ("B", "C", 1), ("C", "D", 1)]
shortest_path_tree_example = [("A", "B", 2), ("A", "C", 2), ("A", "D", 2)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
draw_weighted_graph(contrast_edges, pos=contrast_pos, title="Same weighted graph", ax=axes[0])
draw_weighted_graph(contrast_edges, pos=contrast_pos, tree_edges=mst_example, title=f"MST candidate (weight {total_weight(mst_example)})", ax=axes[1])
draw_weighted_graph(
    contrast_edges,
    pos=contrast_pos,
    tree_edges=shortest_path_tree_example,
    title=f"Tree with shortest routes from A (weight {total_weight(shortest_path_tree_example)})",
    ax=axes[2],
)
plt.show()

print("Route lengths measured inside each finished tree from A:")
print("MST candidate:", route_lengths_in_tree(mst_example, "A"))
print("Shortest-path-style tree:", route_lengths_in_tree(shortest_path_tree_example, "A"))


## Real Applications of MST

Good undergraduate-level examples include:
- building a low-cost network of cables, pipes, or roads,
- designing a cheap backbone before adding redundancy,
- clustering data by building an MST and then cutting a few heavy edges,
- approximation algorithms on metric graphs, for example as a step toward TSP approximations.

Important practical note:
an MST gives a **cheap** connected network, but not necessarily a **robust** one. If an edge fails, a tree has no backup cycle.


## Cross-check with NetworkX

For experimentation in Colab, `networkx` is convenient and already includes MST implementations for Kruskal, Prim, and Boruvka.


In [ ]:
nx_graph = build_graph(sample_edges)

for algorithm in ("kruskal", "prim", "boruvka"):
    nx_tree = nx.minimum_spanning_tree(nx_graph, algorithm=algorithm)
    tree_edges = [
        (edge_key(u, v)[0], edge_key(u, v)[1], data["weight"])
        for u, v, data in nx_tree.edges(data=True)
    ]
    print_tree(f"NetworkX {algorithm}", tree_edges)
    print()


## Summary

Main takeaways:
- A spanning tree connects all vertices without cycles.
- An MST minimizes the total weight of that tree.
- If all edges have equal weight, every spanning tree is minimum.
- Kruskal grows a forest by taking the lightest safe edges.
- Prim grows one tree by repeatedly taking the cheapest frontier edge.
- Ties can create multiple valid MSTs.
- MST and shortest-path trees solve different problems.


## Verified References

These links were checked while refactoring this notebook on April 24, 2026.

1. Jeff Erickson, *Algorithms*, Chapter 7 "Minimum Spanning Trees"  
   https://jeffe.cs.illinois.edu/teaching/algorithms/

2. MIT OpenCourseWare, Lecture 12 "Greedy Algorithms: Minimum Spanning Tree"  
   https://ocw.mit.edu/courses/6-046j-design-and-analysis-of-algorithms-spring-2015/resources/lecture-12-greedy-algorithms-minimum-spanning-tree/

3. Joseph B. Kruskal, "On the shortest spanning subtree of a graph and the traveling salesman problem" (1956), American Mathematical Society  
   https://www.ams.org/jourcgi/jour-getitem?pii=S0002-9939-1956-0078686-7

4. R. C. Prim, "Shortest Connection Networks and Some Generalizations" (1957), Bell Labs / Nokia  
   https://www.nokia.com/bell-labs/publications-and-media/publications/shortest-connection-networks-and-some-generalizations/

5. Jaroslav Nesetril, Eva Milkova, Helena Nesetrilova, "Otakar Boruvka on minimum spanning tree problem: translation of both the 1926 papers, comments, history" (2001), DML-CZ  
   https://dml.cz/dmlcz/500413

6. NetworkX documentation for `minimum_spanning_tree`  
   https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.tree.mst.minimum_spanning_tree.html

7. Cormen, Leiserson, Rivest, Stein, *Introduction to Algorithms*, 4th edition, MIT Press (2022)  
   https://mitpress.mit.edu/9780262046305/introduction-to-algorithms/
